In [1]:
%pip install gensim

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 23.2.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

from gensim.models import Word2Vec, KeyedVectors
import gensim.downloader as api

#Configurações para melhorar os gráficos
plt.style.use('ggplot')

In [ ]:
# Nossos corpus de brinquedo (lista de lista de tokens)
sentences = [
    ["gato", "come", "peixe"],
    ["cachorro", "come", "carne"],
    ["gato", "e", "cachorro", "sao", "animais"],
    ["peixe", "vive", "na", "agua"],
    ["cachorro", "late"],
    ["gato", "mia"],
    ["leao", "come", "carne"],
    ["leao", "e", "um", "animal", "selvagem"],
    ["peixe", "nada", "na", "agua"]
]

# Treinando o modelo Word2Vec usando CBOW (sg=0)
model = Word2Vec(
    sentences=sentences,
    vector_size=20,     # Vetores menores aqui por ser um corpus minusculo
    window=2,           # Contexto de 2 palavras para trás e 2 para frente
    min_count=1,        # Mantém todas as palavras
    sg=0,               # 0 = CBOW, 1 = Skip-gram
    epochs=200          # Número de iterações sobre o corpus
)

print("Vocabulário aprendido:", list(model.wv.index_to_key))



Vocabulário aprendido: ['cachorro', 'peixe', 'come', 'gato', 'leao', 'agua', 'na', 'e', 'carne', 'nada', 'selvagem', 'animal', 'um', 'mia', 'late', 'vive', 'animais', 'sao']


# 2. Acesso a Vetores e Propriedades do Modelo

Uma vez treinamento, os vetores ficam armazenado no objeto model.wv (Word Vectors). Este objeto é do tipo KeyedVectors e é otimizado para buscar rápidas.

In [6]:
# Acessando a representação densa (vetor) de uma palavra
vetor_gato = model.wv["gato"]
print("Dimensao do vetor:", vetor_gato.shape)
print("Vetor de 'gato':\n", vetor_gato)

Dimensao do vetor: (20,)
Vetor de 'gato':
 [-0.00840339  0.00164755 -0.0204895  -0.03837592 -0.0073668   0.01291641
 -0.00305661  0.0287989  -0.0144895   0.0116235   0.0290909   0.04076406
 -0.00804086 -0.04687313  0.02164288  0.00329518  0.03922033 -0.00363401
 -0.01423957 -0.04395481]


# 3. Explorando o Espaço Semântico

Como o modelo mapeou as palavras para um espaço vetorial, podemos usar a Similaridade de Cosseno para encontrar a distância angular entre dois vetores.
Quanto mais próximo de 1, mais similares são os contextos em que as palavras aparecem.

In [ ]:
# Calculando a similaridade de cosseno diretamente entre duas palavras
similaridade_gato_cachorro = model.wv.similarity("gato", "cachorro")
similaridade_gato_peixe = model.wv.similarity("gato", "peixe")

print(f"Similaridade (gato, cachorro): {similaridade_gato_cachorro:.4f}")
print(f"Similaridade (gato, peixe): {similaridade_gato_peixe:.4f}\n")

# Encontrando as N palavras mais similares no espaco vetorial
print("Palabras mais similares a 'come':")
for palavra, score in model.wv.most_similar("come", topn=3):
    print(f" - {palavra} ({score:.4f})")

# ODD-ONE-OUT: Qual palavra nao pertence ao grupo?
print("\nPalavra intrusa no grupo ['gato', 'cachorro', 'carne']: ",
    model.wv.doesnt_match(["gato", "cachorro", "carne"]))

Similaridade (gato, cachorro): 0.09
Similaridade (gato, peixe): 0.16

Palavras mais similares a 'come':
 - vive (0.39)
 - gato (0.24)
 - selvagem (0.21)

Palavra intrusa no grupo ['gato', 'cachorro', 'carne']:  carne


# 4. Álgebra Vetorial com Palavras (Analogias)

Uma das propriedades mais fascinantes dos Embeddings é a capacidade de realizar operações matemáticas com palavras. O exemplo clássico da literatura é:

                                                    Vetor(Rei) - Vetor(Homem) + Vetor(Mulher) ~ Vetor(Rainha)

No Gensim, fazemos isso usando o método most_similar combinando os argumentos positive (que são somados) e negative (que são subtraídos).

In [16]:
# Tentando uma analogia no nosso corpus minúsculo:
# Se 'cachorro' esta para 'carne', assim como 'peixe' está para o que?
# Equação: X = carne - cachorro + peixe
result = model.wv.most_similar(positive=["carne", "peixe"], negative=["cachorro"], topn=2)
for palavra, score in result:
    #print(f"X = carne - cachorro + peixe: {palavra} com similiaridade: {score}")
    print(f"carne - cachorro + peixe = {palavra:8} | {score:.2f}")

carne - cachorro + peixe = selvagem | 0.30
carne - cachorro + peixe = late     | 0.28


Onde se similaridade for:

- 1.0: Palavras idênticas ou sinônimos perfeitos.

- 0.0: Palavras sem relação.

- -1.0: Significados opostos (raro em embeddings comuns).

# 5. Salvando e Carregando Modelos

Treinar embeddings em bilhões de palavras pode levar dias. Por isso, precisamos salvar os modelos. O Gensim permite salvar o modelo completo (permitindo
continuar o treinamento depois) ou apenas os vetores (KeyedVectors), que ocupa muito menos memória.

In [ ]:
# 1. Salvando apenas os vetores (Recomendado para deploy/producão)
model.wv.save("meus_vetores.kv")
wv_carregado = KeyedVectors.load("meus_vetores.kv")
print("Vetor recuperado do disco com sucesso.")

# 2. Salvando o modelo completo (Recomendado se for continuar o treinamento)
model.save("meu_word2vec.model")

Vetor recuperado do disco com sucesso.


In [18]:
import gensim.downloader as api

# Mostrando os modelos disponíveis
print("Alguns modelos disponíveis para download:")
info = api.info()
for model_name in list(info['models'].keys())[:5]:
    print(f" - {model_name}")

# Carregando um modelo de demonstração rápido (GloVe treinado no Twitter, dimensão 25)
print("\nBaixando/Carregando modelo 'glove-twitter-25'...")
glove_model = api.load("glove-twitter-25")

print("\nSimilaridade entre 'computer' e 'programming' no GloVe:",
    glove_model.similarity("computer", "programming") )

print("\nAnalogia no GloVe: King - Man + Woman =")
for p, s in glove_model.most_similar(positive=['king', 'woman'], negative=['man'], topn=3):
    print(f" - {p} ({s:.4f})")

Alguns modelos disponíveis para download:
 - fasttext-wiki-news-subwords-300
 - conceptnet-numberbatch-17-06-300
 - word2vec-ruscorpora-300
 - word2vec-google-news-300
 - glove-wiki-gigaword-50

Baixando/Carregando modelo 'glove-twitter-25'...
[==================================================] 100.0% 104.8/104.8MB downloaded

Similaridade entre 'computer' e 'programming' no GloVe: 0.70801824

Analogia no GloVe: King - Man + Woman =
 - meets (0.8842)
 - prince (0.8322)
 - queen (0.8257)
